In [ ]:
%pip install pandas numpy scikit-learn tensorflow joblib

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow import keras

In [ ]:
df = pd.read_csv('./placementdata.csv')
df.head()

In [ ]:
# Encode categorical columns
le = LabelEncoder()
for col in ['ExtracurricularActivities', 'PlacementTraining', 'PlacementStatus']:
    df[col] = le.fit_transform(df[col])

df = df.drop(columns=['StudentID'])
df.head()

In [ ]:
# Feature Engineering
df['CGPA_x_Aptitude']            = df['CGPA'] * df['AptitudeTestScore']
df['Total_Marks']                = df['SSC_Marks'] + df['HSC_Marks']
df['Training_x_Extracurricular'] = df['PlacementTraining'] * df['ExtracurricularActivities']
df['CGPA_x_Projects']            = df['CGPA'] * df['Projects']
df['Aptitude_x_SoftSkills']      = df['AptitudeTestScore'] * df['SoftSkillsRating']

print('Features after engineering:', df.shape[1] - 1)
df.head()

In [ ]:
X = df.drop(columns=['PlacementStatus'])
y = df['PlacementStatus']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),

    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(64, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),

    keras.layers.Dense(32, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.1),

    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

print(f'ANN Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['NotPlaced', 'Placed']))

In [ ]:
# Save model and scaler for the Streamlit app
model.save('placement_ann_model.keras')
joblib.dump(scaler, 'placement_scaler.pkl')
print('Model and scaler saved.')

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()